# Laboratory 1 - Introduction and Knowledge Distillation

This is the first laboratory of the Embedded AI course. Its job is to get everyone on the same page before we move on to more specialized topics.

We will go through four things today:

0. A short setup guide, so you have a working Python environment for this and the following laboratories.
1. A PyTorch recap. If you already know PyTorch well, skim it. If you have never used it, read it carefully, everything else in this course builds on it.
2. Why model size is such a big deal on embedded devices, and how to reason about it.
3. Knowledge distillation, a technique that helps a small model perform closer to a big one.

A short word on why this course matters. Most deep learning research happens on GPUs with plenty of memory and power to spare. Most real devices in the world are not GPUs, they are microcontrollers, small SoCs and boards with tight memory, limited compute and often a battery budget. Getting a model to run well on that kind of hardware is a genuinely different problem from getting it to run well on a server, and it needs a different set of tools. This course is about those tools, and today we start with the basics plus the first of them: knowledge distillation.

**What you should already know:** basic Python, and roughly what a neural network, a loss function and gradient descent are.

**What you will be able to do after this notebook:** set up a working PyTorch environment on your own machine, write a training loop from scratch and explain what each part does, estimate whether a model fits on a given embedded device, and implement knowledge distillation.


## 0. Setup

You need Python 3.10 or newer and a handful of packages. Set up an isolated environment for this so the course does not interfere with anything else on your machine, and so your setup is reproducible.

**Conda**
```
conda create -n embedded-ai python=3.10 -y
conda activate embedded-ai
```

**Or plain venv, if you don't use conda**
```
python -m venv .venv
source .venv/bin/activate      # Linux / macOS
.venv\Scripts\activate         # Windows
```

Then install PyTorch. The right command depends on your OS and on whether you have a CUDA-capable GPU, so use the official selector instead of copying a fixed command from somewhere: https://pytorch.org/get-started/locally/ . If you don't have a GPU or are not sure, the CPU-only build works fine for this notebook, it is just slower.

`torchvision` is installed together with `torch` by the command above. You also need matplotlib and gdown, the second one only fetches the dataset for us in the next section:
```
pip install matplotlib gdown
```

Packages used in this notebook, for reference:
- torch >= 2.0
- torchvision >= 0.15
- matplotlib
- gdown

Run the cell below. It imports everything we need and checks whether a GPU is available. If it runs without errors, you're set for the rest of this laboratory (and most of the following ones).

In [ ]:
import torch
import torchvision
import matplotlib
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("matplotlib version:", matplotlib.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(0)

## 1. PyTorch recap

This section is a refresher, not a full introduction to PyTorch. If this is your first time using it, take it slowly and run every cell, it is worth it. If you already know PyTorch well, skim through and jump to Section 2.

Every PyTorch project in this course follows the same pattern: data, model, loss and optimizer, and a training loop that ties them together. We build all four pieces below and reuse them for the rest of the notebook.

### 1.1 Data

We use CIFAR-10: 60,000 small color images (32x32 pixels) in 10 classes such as airplane, cat or truck, split into 50,000 training and 10,000 test images. It is small enough to train quickly on a laptop GPU, but not so simple that every model reaches 99 percent accuracy in a minute.

`torchvision` can download CIFAR-10 for you automatically, but the official mirror is sometimes very slow depending on where you are. The cell below instead fetches it from a Drive mirror with `gdown`, which is usually much faster, and only does so if the data is not already sitting in the `data` folder.

In [ ]:
import os
import zipfile

data_dir = "data"
if not os.path.isdir(os.path.join(data_dir, "cifar-10-batches-py")):
    print("CIFAR-10 not found locally, downloading...")
    import gdown
    zip_path = "cifar10.zip"
    gdown.download(id="1b7UF-Wu6aiOUhy4W2Yl9_1cO4SSApAPn", output=zip_path, quiet=False)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(".")
    os.remove(zip_path)
else:
    print("CIFAR-10 already present locally, skipping download.")

`transforms.Normalize` rescales each color channel so pixel values are roughly centered around zero, which makes training more stable. The numbers below are the mean and standard deviation of the CIFAR-10 training set, the commonly used values for this dataset.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_set = datasets.CIFAR10("data", train=True, download=True, transform=transform)
test_set = datasets.CIFAR10("data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000)

classes = train_set.classes
print("Training images:", len(train_set))
print("Test images:", len(test_set))
print("Classes:", classes)

In [ ]:
# A quick look at a few images. We have to undo the normalization to display them.
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    image, label = train_set[i]
    ax.imshow((image * std + mean).permute(1, 2, 0).clamp(0, 1))
    ax.set_title(classes[label], fontsize=8)
    ax.axis("off")
plt.show()

### 1.2 Defining a model

A PyTorch model is a class that inherits from `nn.Module`. In `__init__` you create the layers you will use, and in `forward` you describe how data flows through them.

When a network is simple, meaning data just goes from one layer straight into the next with nothing branching off, `nn.Sequential` is a convenient shortcut: it chains a list of layers for you so you don't have to call each one by hand in `forward`. We use it below.

You will build a small convolutional network yourself: three blocks of convolution, batch normalization, ReLU and max pooling, followed by one linear layer that turns the final feature maps into 10 numbers, one per class. These raw numbers are called **logits**, they only become probabilities after a softmax.

A quick reminder of what each layer in a block does:

- **Convolution** (`nn.Conv2d`) slides a small learnable filter, called a kernel, across the image. Each filter reacts to some local pattern, an edge, a corner, a color blob, and stacking several filters lets the layer pick up on many patterns at once. With a 3x3 kernel and `padding=1`, the output has the same height and width as the input, only the number of channels changes.
- **Batch normalization** (`nn.BatchNorm2d`) rescales the activations of a layer so they stay in a reasonable range as they flow through the network, which generally makes training faster and more stable.
- **ReLU** (`nn.ReLU`) is the activation function: it simply replaces every negative value with zero. Without a non-linearity like this, stacking layers would be mathematically equivalent to using just one layer.
- **Max pooling** (`nn.MaxPool2d(2)`) shrinks the height and width by a factor of 2, keeping only the strongest activation in each small window. It reduces the amount of computation and makes the network care a little less about the exact position of a feature.

If any of this is new, or you want the details, the PyTorch documentation is the right place to look:
- Conv2d: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- BatchNorm2d: https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html
- ReLU: https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html
- MaxPool2d: https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
- Linear: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
- Sequential: https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html

And if convolutions as a concept are unfamiliar, this is a good general explanation, not PyTorch specific: https://cs231n.github.io/convolutional-networks/

Now build `conv_block`, one building block as described above, and `SmallCNN`, which chains three of them together (channels going 3 -> width -> 2*width -> 4*width) and finishes with a flatten and a single linear layer down to 10 classes. Work out the number of input features for that linear layer yourself: the image starts at 32x32, and each of the three blocks halves both dimensions.

In [ ]:
def conv_block(in_channels, out_channels):
    """
    One building block: a 3x3 convolution that keeps the spatial size the
    same, followed by batch norm, ReLU, and a max pool that halves the
    spatial size. Return the four layers as a list.
    """
    # TODO: implement, see the layer reminders above
    raise NotImplementedError


class SmallCNN(nn.Module):
    def __init__(self, width=16):
        super().__init__()
        # TODO: build self.features by chaining three conv_block calls.
        # The channel count should go 3 -> width -> 2*width -> 4*width.

        # TODO: build self.classifier: a Flatten followed by a single
        # Linear layer mapping down to 10 classes. Figure out how many
        # input features the Linear layer needs given a 32x32 input image
        # and three blocks that each halve height and width.
        raise NotImplementedError

    def forward(self, x):
        # TODO: pass x through self.features, then self.classifier
        raise NotImplementedError


model = SmallCNN(width=16)
print(model)

In [ ]:
# A quick sanity check: one batch of 4 images should give 4 x 10 logits
dummy = torch.randn(4, 3, 32, 32)
print("Output shape:", model(dummy).shape)

### 1.3 Training and evaluation

The training loop does the same five things, over and over:

1. take a mini-batch of data,
2. run it through the model (forward pass),
3. compute the loss,
4. compute gradients with `loss.backward()`,
5. update the weights with `optimizer.step()`.

Two details that are easy to forget the first time you write this:

- `optimizer.zero_grad()` has to be called every step, otherwise gradients from the previous step get added on top of the new ones.
- `model.train()` and `model.eval()` switch the behavior of layers like batch norm and dropout. Use `eval()` together with `torch.no_grad()` whenever you only want predictions, not training.

Let's start with evaluation, since the training loop below will use it to report progress after every epoch.

In [ ]:
def evaluate(model, loader):
    """
    Compute the classification accuracy of `model` on all the batches in `loader`.

    Put the model in evaluation mode first, and make sure no gradients are
    computed, we are only doing inference here.
    """
    # TODO: implement this function. It should return a float between 0 and 1.
    raise NotImplementedError

In [ ]:
def train(model, epochs, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)                    # forward pass
            loss = F.cross_entropy(outputs, labels)    # loss

            optimizer.zero_grad()
            loss.backward()                            # backward pass
            optimizer.step()                           # weight update

        print(f"epoch {epoch + 1}/{epochs}  last batch loss: {loss.item():.3f}  "
              f"test accuracy: {evaluate(model, test_loader):.3f}")

In [ ]:
# Try it on our small model for just 2 epochs, to see that everything works
demo_model = SmallCNN(width=16)
train(demo_model, epochs=2)

That's the whole recap. From here on, whenever we say "train the model" we mean this loop, sometimes with a different loss (we will change that in Section 3).

One practical note: `train` checks the test accuracy after every epoch just so you can see progress while working through this notebook. In a real project you should tune hyperparameters on a separate validation set and only look at the test set at the very end, otherwise you slowly overfit to it without noticing.

## 2. Why model size matters on embedded systems

On a PC or a server we rarely think about how big a model is. On a microcontroller we have to, because memory is tiny and it is usually the first thing that decides whether a model can run at all.

### 2.1 Two kinds of memory

A typical microcontroller has two kinds of memory that matter to us:

- **Flash** (non-volatile): stores the program code and constant data. The trained **weights** of the network normally live here, because they don't change during inference.
- **RAM** (SRAM): working memory. During inference it holds the **activations**, the intermediate outputs of the layers, plus the stack and buffers for the input data.

So a model has to fit in two places at once: the weights in flash, and the activations in RAM. Bigger devices, such as a Raspberry Pi or a Jetson, have gigabytes of RAM, and there the concern shifts from "does it fit" to speed and energy. We'll get to that at the end of this section.

### 2.2 How to estimate the weight size

The weight memory is simple to estimate:

```
size in bytes = number of parameters x bytes per parameter
```

With the default 32-bit floating point (float32) each parameter takes 4 bytes. With 8-bit integers (int8), which we will meet later in the course under the name quantization, it takes 1 byte. So just changing the number format makes a model roughly 4 times smaller, with no change to the architecture at all.

Write two small helper functions: one that counts the parameters of a model, and one that turns that count into a size in kilobytes.

In [ ]:
def count_parameters(model):
    """Return the total number of trainable parameters in `model`."""
    # TODO: implement. Look at what model.parameters() gives you, and at
    # what a tensor's .numel() returns.
    raise NotImplementedError


def model_size_kb(model, bytes_per_param=4):
    """
    Return the size of `model`'s weights in kilobytes, assuming every
    parameter takes `bytes_per_param` bytes (4 for float32, 1 for int8).
    """
    # TODO: implement, reuse count_parameters above
    raise NotImplementedError


small = SmallCNN(width=16)
print("Parameters:", count_parameters(small))
print(f"Size as float32: {model_size_kb(small, 4):.1f} KB")
print(f"Size as int8:    {model_size_kb(small, 1):.1f} KB")

### 2.3 Some well known models

To get a feeling for the numbers, here are a few standard image classification networks from `torchvision`. We create them without pretrained weights (`weights=None`), we only need the architecture here to count parameters, not to actually classify anything.

In [ ]:
known_models = {
    "MobileNetV2":     models.mobilenet_v2(weights=None),
    "ResNet-18":       models.resnet18(weights=None),
    "ResNet-50":       models.resnet50(weights=None),
    "our SmallCNN":    SmallCNN(width=16),
}

print(f"{'model':<14}{'parameters':>14}{'float32 [MB]':>16}{'int8 [MB]':>12}")
for name, m in known_models.items():
    n = count_parameters(m)
    print(f"{name:<14}{n:>14,}{n * 4 / 1024**2:>16.2f}{n / 1024**2:>12.2f}")

### 2.4 Embedded platforms

The numbers below are for small microcontrollers, where flash and RAM are measured in kilobytes and every kilobyte matters:

- **Arduino Uno** (ATmega328P): a very old and very constrained 8-bit microcontroller, useful here mostly as a lower bound of what "tiny" means.
- **STM32F407**: a common 32-bit ARM Cortex-M4 microcontroller, a realistic target for lightweight TinyML.
- **ESP32-P4**: a newer RISC-V based chip from Espressif aimed at edge AI, with noticeably more SRAM than the STM32 and no built-in flash, it always uses external flash (boards typically ship with several MB to tens of MB of it).

Exact numbers depend on the exact chip revision and board, so always check the datasheet of the board you actually have before trusting a table like this one.

There is also a bigger class of embedded devices: boards like the **Raspberry Pi 5** (1 to 16 GB of RAM depending on the model), the **Jetson Orin Nano** (4 or 8 GB, plus a small GPU meant for running neural networks efficiently), or **Kria** FPGA modules (a few GB of DDR memory next to reconfigurable FPGA fabric). These run a full Linux system, and a model that is a few megabytes in size is rarely a problem there. What matters instead is inference speed, power draw, and for the Kria, how well the model maps onto the FPGA fabric. We'll come back to platforms like these later in the course, for now the table below focuses on the tight, KB-scale end of the spectrum, where the size question is the sharpest.

In [ ]:
# name: (flash in KB, RAM in KB)
platforms = {
    "Arduino Uno (ATmega328P)": (32, 2),
    "STM32F407":                (1024, 192),
    "ESP32-P4 (16 MB ext. flash)": (16384, 768),
}

print(f"{'platform':<30}{'flash [KB]':>12}{'RAM [KB]':>10}")
for name, (flash, ram) in platforms.items():
    print(f"{name:<30}{flash:>12}{ram:>10}")

In [ ]:
def fits_in_flash(model, flash_kb, bytes_per_param=4):
    return model_size_kb(model, bytes_per_param) < flash_kb


for name, (flash, ram) in platforms.items():
    for label, m in known_models.items():
        f32 = "yes" if fits_in_flash(m, flash, 4) else "no"
        i8 = "yes" if fits_in_flash(m, flash, 1) else "no"
        print(f"{name:<30}{label:<14} float32: {f32:<4} int8: {i8}")
    print()

### 2.5 What this estimate does not include

The numbers above are a **lower bound**, not the whole story:

- Flash also has to hold the program itself: the inference code, the runtime, and so on.
- We haven't checked RAM at all. The activations of the largest layer, plus the input buffer, have to fit in RAM. For convolutional networks, the early layers with large feature maps are often the real bottleneck, not the weights.
- Some platforms store weights with extra metadata or in a different layout.

Still, the parameter count is a good first filter. If the weights alone don't fit, there is no point looking further.

This is the motivation for the rest of the lecture: we want models with **fewer parameters that keep as much accuracy as possible**. There are several tools for that: efficient architectures, pruning, quantization, and knowledge distillation. Today we focus on the last one.

## 3. Knowledge distillation

### 3.1 The idea

Big models usually reach better accuracy than small ones, but we can't put them on the device. Small models fit, but they tend to be less accurate when trained the usual way.

Knowledge distillation (Hinton, Vinyals and Dean, 2015, "Distilling the Knowledge in a Neural Network", arXiv:1503.02531) proposes the following: first train a large **teacher** network. Then train a small **student** network to imitate the teacher, in addition to learning from the labels directly.

Why would imitating the teacher help? Think of a picture of a cat. The correct label is "cat", and as a vector that is `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`. It tells the student nothing about how the classes relate to each other. But the teacher's output for this image might say something like "cat: 90%, dog: 8%, truck: close to 0%". That tells the student this particular cat looks a bit like a dog and nothing like a truck. This kind of information about similarity between classes is sometimes called **dark knowledge**, and it simply isn't present in the hard labels.

Important for embedded systems: the teacher is used **only during training**, on a big machine. After training we deploy only the student, so the device pays nothing extra for having had a teacher.

### 3.2 Soft targets and temperature

The network outputs logits `z`. The normal softmax turns them into probabilities:

```
p_i = exp(z_i) / sum_j exp(z_j)
```

A well-trained teacher is usually very confident, so its output is close to `[0, 0, 0, 0.99, 0, ...]`, and the small probabilities that actually carry the useful information are almost zero. To make them visible we use a **temperature** `T`:

```
p_i(T) = exp(z_i / T) / sum_j exp(z_j / T)
```

With `T = 1` we get the normal softmax. With `T > 1` the distribution becomes softer, flatter. Let's see it on an example.

In [ ]:
# Example logits of some network for 3 classes
logits = torch.tensor([8.0, 2.0, 1.0])

for T in [1, 2, 4, 8]:
    probs = F.softmax(logits / T, dim=0)
    print(f"T = {T}:  {[round(p, 4) for p in probs.tolist()]}")

With `T = 1` almost all the probability sits on the first class. With higher `T` the other classes become visible, while the relative order between them stays the same.

### 3.3 The loss function

The student is trained with a combination of two losses:

1. **The usual cross-entropy** between the student's output and the true label, the "hard" loss.
2. **A distillation loss** measuring how far the student's softened output is from the teacher's softened output, the "soft" loss. We use the Kullback-Leibler (KL) divergence for this. Both outputs are computed with the same temperature `T`.

```
loss = alpha * T^2 * KL( softmax(teacher / T) || softmax(student / T) ) + (1 - alpha) * CE(student, label)
```

Two things worth explaining:

- `alpha`, between 0 and 1, says how much we trust the teacher compared to the labels. `alpha = 0` gives back normal training.
- The `T^2` factor is there because the gradients of the soft loss shrink by roughly `1 / T^2` as the temperature grows. Multiplying by `T^2` keeps the two loss terms at a comparable scale, so you don't have to retune `alpha` every time you change `T`. This is also mentioned in the original paper.

`T` and `alpha` are hyperparameters. Typical choices are `T` between 2 and 6 and `alpha` between 0.3 and 0.7, but the best values depend on the problem.

### 3.4 In PyTorch

One detail to watch: `F.kl_div` expects the first argument as **log-probabilities** and the second as **probabilities**. That's why the student side uses `log_softmax` and the teacher side uses `softmax`. The option `reduction="batchmean"` averages over the batch, which is the mathematically correct way to compute the KL divergence here.

Implement the distillation loss below, following the formula from 3.3.

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    """
    Combine the soft distillation loss (student vs. softened teacher output)
    with the normal hard cross-entropy loss (student vs. true labels), as
    described in 3.3:

        loss = alpha * T^2 * KL(soft_teacher || soft_student) + (1 - alpha) * CE(student, labels)

    """
    # TODO: implement. You will need F.log_softmax, F.softmax, F.kl_div
    # (mind the argument order) and F.cross_entropy.
    raise NotImplementedError

### 3.5 The experiment

We want to answer one question: **does the same small network get better if it is trained with a teacher?**

The setup:

- **Teacher**: a ResNet-18, finetuned from ImageNet weights (more on this in a moment).
- **Student (baseline)**: our `SmallCNN` from Section 1, trained normally with cross-entropy.
- **Student (KD)**: exactly the same architecture and the same initialization, trained with the distillation loss instead.

The only difference between the two students is the loss function. That matters, otherwise we wouldn't be able to say what actually caused any difference we see.

In [ ]:
TEACHER_EPOCHS = 10   # the teacher starts from pretrained weights, but a few more epochs make it noticeably more confident
STUDENT_EPOCHS = 15   # both students train from scratch, so they need more

### 3.6 The teacher: finetuning a pretrained ResNet-18

Before training the teacher, a short word on finetuning, since that's what we use here.

Training a network like ResNet from scratch takes a long time and a lot of data. Finetuning skips most of that: you start from a network that was already trained on a large, general dataset, ImageNet in our case, and adapt it to your own, smaller dataset. The early layers of a trained CNN tend to learn fairly general things, edges, textures, simple shapes, that are useful for almost any image task. The later layers are more specific to the original classes. So you can keep most of the network as it is and only retrain the parts that need to change, which is usually much faster and needs far less data than training from zero.

For us specifically: `torchvision` ships a ResNet-18 with ImageNet weights (1000 classes). CIFAR-10 has only 10 classes and much smaller images, 32x32 instead of 224x224, so two things need to change before we can finetune it here: the final layer needs to output 10 classes instead of 1000, and the very first layers need adjusting so they don't shrink a 32x32 image away to almost nothing before the network has done any real work.

Your task: create a pretrained ResNet-18, then adapt it for CIFAR-10.

In [ ]:
# TODO: create a ResNet-18 with pretrained ImageNet weights.
# Look at torchvision.models.resnet18 and models.ResNet18_Weights.
teacher = ...

# TODO: replace teacher.conv1 with a convolution better suited to small
# 32x32 images. Think about what a 7x7 stride-2 conv followed by a max pool
# does to an image that small, and what a gentler alternative would look like.

# TODO: get rid of the max pool that immediately follows the stem
# (hint: a layer that does nothing to its input still needs to be a module).

# TODO: replace teacher.fc so the network outputs 10 classes instead of 1000.
# The existing layer tells you how many input features to keep.

print("Teacher parameters:", count_parameters(teacher))

Since most of the network is already trained, a smaller learning rate than usual works better here, we don't want to wreck the pretrained features in the first few batches. `train` already takes an `lr` argument, so we just pass a smaller one.

In [ ]:
train(teacher, epochs=TEACHER_EPOCHS, lr=1e-4)
teacher_acc = evaluate(teacher, test_loader)
print("Teacher test accuracy:", teacher_acc)

### 3.7 The students: baseline vs. knowledge distillation

First, the student trained the normal way. We reset the seed before creating each student, so both start from the same initial weights, that's the only fair way to compare them.

In [ ]:
torch.manual_seed(1)
student_baseline = SmallCNN(width=16)
print("Student parameters:", count_parameters(student_baseline))

train(student_baseline, epochs=STUDENT_EPOCHS)
baseline_acc = evaluate(student_baseline, test_loader)
print("Student (no KD) test accuracy:", baseline_acc)

Now the distillation version. Write `train_with_kd`, a version of `train` from Section 1 with two changes: on every batch it also runs the teacher (with gradients turned off, the teacher isn't being trained), and it uses `distillation_loss` instead of plain cross-entropy. Everything else, the optimizer, the epoch loop, the progress printing, stays the same.

In [ ]:
def train_with_kd(student, teacher, epochs, T=4.0, alpha=0.5, lr=1e-3):
    """
    Like train() from Section 1, but the student learns from a mix of the
    labels and the teacher's softened predictions, using distillation_loss
    instead of plain cross-entropy.

    The teacher is not being trained here: make sure it stays in eval mode
    and that no gradients are tracked through it.
    """
    # TODO: implement, using train() above as your starting point.
    raise NotImplementedError

In [ ]:
torch.manual_seed(1)   # same seed as the baseline -> same initial weights
student_kd = SmallCNN(width=16)

train_with_kd(student_kd, teacher, epochs=STUDENT_EPOCHS, T=6.0, alpha=0.7)
kd_acc = evaluate(student_kd, test_loader)
print("Student (with KD) test accuracy:", kd_acc)

### 3.8 Results

In [ ]:
results = {
    "Teacher": (teacher, teacher_acc),
    "Student, no KD": (student_baseline, baseline_acc),
    "Student, with KD": (student_kd, kd_acc),
}

print(f"{'model':<20}{'parameters':>12}{'size [KB]':>12}{'accuracy':>12}")
for name, (m, a) in results.items():
    print(f"{name:<20}{count_parameters(m):>12,}{model_size_kb(m):>12.0f}{a:>12.3f}")

A note on how to read a comparison like this one. We trained each model only once, so the gap between the two students is partly random, a different seed would give slightly different numbers. A difference of a few tenths of a percent is well within that noise. To make a solid claim you would repeat the experiment with several seeds and compare the averages, which is exactly one of the exercises below.

### 3.9 Exercises

1. **Different seeds.** Repeat the baseline and the KD training for 3 different seeds and compute the mean accuracy of each. Is the improvement consistent, or does it disappear in the noise?
2. **Temperature.** Train the KD student with `T = 1`, `T = 2` and `T = 8`. What changes? Why would `T = 1` be less useful for distillation than a higher value?
3. **Alpha.** Try `alpha = 0.1` and `alpha = 0.9`. What does `alpha = 0` correspond to?
4. **Student size.** Change the student's `width` to 8 or 32. Does KD help more for a very small student, or for a bigger one?
5. **Model size on a real device.** Use `model_size_kb` to compute the size of the student with int8 weights. Which of the platforms from Section 2.4 could hold it?
6. **Freezing layers.** Right now we finetune the whole ResNet-18. Try freezing everything except the last block and the classification head (`requires_grad = False` on the parameters you want frozen), and compare training time and final accuracy to the fully finetuned version.

### 3.10 Common pitfalls

- Forgetting `teacher.eval()`. With the teacher in training mode, dropout and batch norm behave differently and the teacher's targets become noisy.
- Forgetting `torch.no_grad()` around the teacher's forward pass. The result is still correct, but it wastes memory and time.
- Swapping the arguments of `F.kl_div`. It expects log-probabilities first, probabilities second.
- Using too high a learning rate when finetuning a pretrained model. It's easy to undo most of what the pretrained weights already know in the first epoch.
- Comparing a KD student against a baseline trained for a different number of epochs or with different settings, that's not a fair comparison anymore.

### 3.11 Summary

- Memory is the main hard limit on embedded devices, and weight size is estimated as parameters times bytes per parameter.
- Finetuning lets you reuse a network already trained on a large dataset, adapting only what needs to change for your task, instead of training from scratch.
- In knowledge distillation, a small student learns from the softened outputs of a large teacher in addition to the labels.
- The teacher is only needed during training. The deployed model is just the small student, so KD costs nothing extra at inference time.
- Distillation doesn't always give a large improvement. Its effect depends on the task and on the teacher and student sizes and the chosen hyperparameters, and it should always be checked with a fair, controlled comparison.

### 3.12 Further reading

- G. Hinton, O. Vinyals, J. Dean, *Distilling the Knowledge in a Neural Network*, 2015, arXiv:1503.02531.
- K. He et al., *Deep Residual Learning for Image Recognition*, 2015, arXiv:1512.03385 (the ResNet paper).
- PyTorch tutorials, in particular the transfer learning tutorial: https://pytorch.org/tutorials/
